In [ ]:
import os
import json
import csv
from pathlib import Path
from pprint import pprint

# NLP imports
import nltk
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
from nltk import word_tokenize

# numeric & text
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# plotting & dataframes
import matplotlib.pyplot as plt
import pandas as pd

for pkg in ("wordnet", "omw-1.4", "punkt"):
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        nltk.download(pkg)

# 1. Dataset preparation: queries (>=15) and ambiguous words
queries = [
    ("I need to book a table for two tonight", "book"),
    ("She will book the tickets online", "book"),
    ("The bat flew out of the cave", "bat"),
    ("He bought a new cricket bat", "bat"),
    ("I deposited money in the bank yesterday", "bank"),
    ("The river bank was eroded after the storm", "bank"),
    ("Turn on the light in the hallway", "light"),
    ("That idea is light and refreshing", "light"),
    ("They lit a match to start the campfire", "match"),
    ("The tennis match lasted three hours", "match"),
    ("We saw a spring bubbling from the ground", "spring"),
    ("She will spring into action when needed", "spring"),
    ("The crane lifted the steel beam", "crane"),
    ("I spotted a graceful crane by the lake", "crane"),
    ("The mole dug tunnels under the garden", "mole"),
    ("The mole on her cheek is tiny", "mole"),
    ("Please charge my phone", "charge"),
    ("The officer will press a charge against him", "charge"),
    # add more if you want
]

# Unique ambiguous words in dataset
ambiguous_words = sorted(set(w for _, w in queries))

# 1.b Build sense dictionary from WordNet (all synsets, definitions, examples)             
sense_dict = {}

# Fallback manual dictionary for words not present or when WordNet isn't available
manual_sense_dict = {
    # minimal manual fallbacks if WordNet misses something; keep consistent with WordNet sense ids
    "book": [
        {"synset": "book.n.01", "definition": "A written or printed work; a set of pages.", "examples": ["I read a book."]},
        {"synset": "book.v.01", "definition": "To reserve in advance (e.g., a seat, a room).", "examples": ["I will book a table."]}
    ],
    # add other manual fallbacks as needed...
}

for word in ambiguous_words:
    syns = wn.synsets(word)
    if not syns:
        # fallback to manual if available
        senses = manual_sense_dict.get(word, [])
        sense_dict[word] = senses
    else:
        senses = []
        for s in syns:
            senses.append({
                "synset": s.name(),
                "definition": s.definition(),
                "examples": s.examples()
            })
        sense_dict[word] = senses

# Save dataset (query, ambiguous_word, possible_senses)
out_dir = Path("wsd_outputs")
out_dir.mkdir(exist_ok=True)

dataset_rows = []
for q, w in queries:
    poss = [s["synset"] for s in sense_dict.get(w, [])]
    dataset_rows.append({"query": q, "ambiguous_word": w, "possible_senses": poss})

csv_path = out_dir / "wsd_dataset.csv"
json_path = out_dir / "wsd_dataset.json"

with open(csv_path, "w", newline='', encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["query", "ambiguous_word", "possible_senses"])
    writer.writeheader()
    for r in dataset_rows:
        # write possible_senses as JSON string to preserve list in CSV
        row = r.copy()
        row["possible_senses"] = json.dumps(row["possible_senses"], ensure_ascii=False)
        writer.writerow(row)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(dataset_rows, f, indent=2, ensure_ascii=False)

print(f"Saved dataset: {csv_path}, {json_path}")

# 2. Algorithm implementation
#    - Lesk (NLTK)
#    - Embedding-based similarity (SBERT preferred; TF-IDF fallback)
use_sbert = False
sbert_model = None
try:
    from sentence_transformers import SentenceTransformer
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
    use_sbert = True
    print("Using Sentence-BERT (all-MiniLM-L6-v2) for embeddings.")
except Exception as e:
    print("SentenceTransformers not available or failed to load. Falling back to TF-IDF for embedding similarity.")
    use_sbert = False

def embedding_similarities(context, sense_texts):
    """
    Returns list of similarity scores between context and each sense_text.
    Uses SBERT if available, otherwise TF-IDF.
    """
    if use_sbert and sbert_model is not None:
        # SBERT returns numpy arrays
        ctx_emb = sbert_model.encode([context], convert_to_numpy=True)
        sense_embs = sbert_model.encode(sense_texts, convert_to_numpy=True)
        sims = cosine_similarity(ctx_emb, sense_embs)[0].tolist()
    else:
        # TF-IDF fallback: fit on context + sense_texts to produce comparable vectors
        vec = TfidfVectorizer().fit([context] + sense_texts)
        X = vec.transform([context] + sense_texts).toarray()
        ctx_vec = X[0:1]
        sense_vecs = X[1:]
        sims = cosine_similarity(ctx_vec, sense_vecs)[0].tolist()
    return sims

# Results container
results = []

# Loop over queries and run both WSD methods
for idx, (query_text, amb_word) in enumerate(queries, start=1):
    syns = sense_dict.get(amb_word, [])
    if not syns:
        # no senses available
        results.append({
            "query": query_text,
            "ambiguous_word": amb_word,
            "lesk_pred": None,
            "lesk_def": None,
            "lesk_synset": None,
            "embed_pred": None,
            "embed_def": None,
            "embed_synset": None,
            "agree": False,
            "similarities": []
        })
        continue

    # 1) Lesk (NLTK) - uses WordNet synsets internally if WordNet is available
    try:
        lesk_syn = lesk(word_tokenize(query_text), amb_word)
    except Exception:
        lesk_syn = None

    if lesk_syn:
        lesk_syn_name = lesk_syn.name()
        lesk_def = lesk_syn.definition()
    else:
        lesk_syn_name = None
        lesk_def = None

    # 2) Embedding-based: compute similarity between context and each sense text (definition + examples)
    sense_texts = []
    for s in syns:
        examples = " ".join(s.get("examples", [])) if s.get("examples") else ""
        sense_texts.append(s["definition"] + " " + examples)

    sims = embedding_similarities(query_text, sense_texts)
    best_idx = int(np.argmax(sims))
    embed_syn = syns[best_idx]["synset"]
    embed_def = syns[best_idx]["definition"]

    agree = (lesk_syn_name == embed_syn) if lesk_syn_name else False

    # store similarity info for every candidate
    sim_list = []
    for i, s in enumerate(syns):
        sim_list.append({
            "synset": s["synset"],
            "definition": s["definition"],
            "score": float(sims[i])
        })

    results.append({
        "query": query_text,
        "ambiguous_word": amb_word,
        "lesk_pred": lesk_syn_name,
        "lesk_def": lesk_def,
        "lesk_synset": lesk_syn_name,
        "embed_pred": embed_syn,
        "embed_def": embed_def,
        "embed_synset": embed_syn,
        "agree": agree, 
        "similarities": sim_list
    })

    # Plot similarity bar chart
    labels = [s["synset"] for s in syns]
    scores = sims
    plt.figure(figsize=(8, 4))
    plt.title(f"Similarity scores — '{amb_word}' in: \"{query_text}\"")
    plt.xlabel("Sense (synset)")
    plt.ylabel("Cosine similarity")
    plt.xticks(rotation=45, ha="right")
    plt.bar(labels, scores)
    plt.tight_layout()
    plot_path = out_dir / f"query_{idx}_{amb_word}_similarity.png"
    plt.savefig(plot_path)
    plt.close()

# 3. Experimentation / Summary
#    - Save results JSON
#    - Compare Lesk vs Embedding agreements
#    - Create a summary CSV
results_path = out_dir / "wsd_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# Summary table
summary_rows = []
agree_count = 0
for r in results:
    summary_rows.append({
        "query": r["query"],
        "amb_word": r["ambiguous_word"],
        "lesk_synset": r["lesk_synset"],
        "embed_synset": r["embed_synset"],
        "agree": r["agree"]
    })
    if r["agree"]:
        agree_count += 1

summary_df = pd.DataFrame(summary_rows)
summary_csv = out_dir / "wsd_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"\nWSD completed for {len(results)} queries.")
print(f"Lesk and embedding-based methods agreed on {agree_count} cases.")
print(f"Results JSON: {results_path}")
print(f"Summary CSV: {summary_csv}")
print(f"Dataset: {csv_path}, {json_path}")
print(f"Similarity plots saved in folder: {out_dir}")

# (Optional) print disagreements for quick inspection
print("\nExamples where Lesk and embedding disagree (showing up to 10):")
disagreements = [r for r in results if not r["agree"]]
for d in disagreements[:10]:
    print("Query:", d["query"])
    print("Ambiguous word:", d["ambiguous_word"])
    print("Lesk:", d["lesk_pred"])
    print("Embed:", d["embed_pred"])
    print("---")


[nltk_data] Downloading package wordnet to /Users/sumith/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/sumith/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /Users/sumith/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Saved dataset: wsd_outputs/wsd_dataset.csv, wsd_outputs/wsd_dataset.json


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
